# Puzzle

https://thefiddler.substack.com/p/can-you-hop-around-the-chessboard

### This Week’s Fiddler

A frog is hopping around a chessboard, always from the center of one square to the center of another square. Each square has side length 1, but the board itself is not necessarily 8-by-8. Instead, it’s N-by-N, where N is some large whole number.

Every jump the frog makes must be the same distance, which we’ll call L. The frog wants to make four jumps such that:
* After the fourth jump, the frog has returned to its starting square.
* The frog visits a total of four distinct squares along the way, including the square on which it starts (and also stops).
* The path the frog takes is not a square loop.
* The frog is never on a square that is diagonal (i.e., a bishop’s move away) or horizontal/vertical (i.e., a rook’s move away) from the starting square.

What is the smallest jumping distance L for which this is possible?

### This Week’s Extra Credit

The frog is jumping around the board with the same minimum distance L you just found.

But this time, the frog also wants to be able to hop to every location on the chessboard. What is the minimum value of N for which this is possible?

# Fiddler Solution

I think we need to find a number such that it can be written as the sum of squares in 2 different ways. Then we can use those to construct the sides of a rhombus that will satisfy all the constraints.

Mentally, I calculated that $65 = 8^2 + 1^2 = 7^2 + 4^2$ should do the trick. i.e. L = $\sqrt{65}$.

$50 = 7^2 + 1^2 = 5^2 + 5^2$ is smaller, but won't work because it will end up on a diagonal square.

Let's check that though.

In [139]:
N = 20

sq_table = {}

for i in range(1, N+1):
    for j in range(i, N+1):
        k = i*i + j*j
        if k not in sq_table:
            sq_table[k] = []
        sq_table[k].append((i,j))

for n in sorted(sq_table.keys()):
    if n < N*N: # Only print values we can be sure of.
        pairs = sq_table[n]
        if len(pairs) > 1:
            print(f"{n} can be written as the sum of 2 squares in {len(pairs)} ways: {pairs}")


50 can be written as the sum of 2 squares in 2 ways: [(1, 7), (5, 5)]
65 can be written as the sum of 2 squares in 2 ways: [(1, 8), (4, 7)]
85 can be written as the sum of 2 squares in 2 ways: [(2, 9), (6, 7)]
125 can be written as the sum of 2 squares in 2 ways: [(2, 11), (5, 10)]
130 can be written as the sum of 2 squares in 2 ways: [(3, 11), (7, 9)]
145 can be written as the sum of 2 squares in 2 ways: [(1, 12), (8, 9)]
170 can be written as the sum of 2 squares in 2 ways: [(1, 13), (7, 11)]
185 can be written as the sum of 2 squares in 2 ways: [(4, 13), (8, 11)]
200 can be written as the sum of 2 squares in 2 ways: [(2, 14), (10, 10)]
205 can be written as the sum of 2 squares in 2 ways: [(3, 14), (6, 13)]
221 can be written as the sum of 2 squares in 2 ways: [(5, 14), (10, 11)]
250 can be written as the sum of 2 squares in 2 ways: [(5, 15), (9, 13)]
260 can be written as the sum of 2 squares in 2 ways: [(2, 16), (8, 14)]
265 can be written as the sum of 2 squares in 2 ways: [(3, 1

Looks good. Fiddler answer is square root of 65. Let's plot it as well.

In [140]:
def draw_board(n, pairs, included_char='X', excluded_char=' '):
    board = [[excluded_char for _ in range(n+1)] for _ in range(n+1)]
    for x,y in pairs:
        board[y][x] = included_char
    for row in reversed(board):
        print(' '.join(row))

draw_board(13, pairs=[(0,0), (8,1), (12,8), (4,7)], included_char='@', excluded_char=':')

: : : : : : : : : : : : : :
: : : : : : : : : : : : : :
: : : : : : : : : : : : : :
: : : : : : : : : : : : : :
: : : : : : : : : : : : : :
: : : : : : : : : : : : @ :
: : : : @ : : : : : : : : :
: : : : : : : : : : : : : :
: : : : : : : : : : : : : :
: : : : : : : : : : : : : :
: : : : : : : : : : : : : :
: : : : : : : : : : : : : :
: : : : : : : : @ : : : : :
@ : : : : : : : : : : : : :


# Extra Credit Solution

I can't think of a clever way to solve this except to check things exhaustively. Since we want all points to be reachable, it's fine to start at any position. Starting near the middle would be convenient since it would quickly rule out small boards that don't have enough space for specific moves.

In [141]:
def check_grid(N, moves, wrap_x=False, wrap_y=False, details=False):
    reachable = [[False for _ in range(N)] for _ in range(N)]
    reachable[N//2][N//2] = True
    reachable_count = 1
    pass_count = 0
    while True:
        pass_count += 1        
        newly_reachable_locations = set()
        for x in range(N):
            for y in range(N):
                if reachable[y][x]:
                    for dx, dy in moves:
                        nx, ny = x + dx, y + dy
                        if wrap_x:
                            nx = nx % N
                        if wrap_y:
                            ny = ny % N
                        if 0 <= nx < N and 0 <= ny < N and not reachable[ny][nx]:                                                        
                            newly_reachable_locations.add((nx, ny))
                            if details:
                                print(f"Pass {pass_count}: {reachable_count}: From ({x},{y}) to ({nx},{ny}) using move ({dx},{dy}) with length^2={dx*dx + dy*dy}")
        newly_reachable_count = 0
        for nx, ny in newly_reachable_locations:
            newly_reachable_count += 1
            reachable[ny][nx] = True
        reachable_count += newly_reachable_count
        if newly_reachable_count == 0:
            break
        
    if reachable_count == N*N:
        print(f"{N}x{N}: All cells are reachable.")
        return True
    else:        
        print(f"{N}x{N}: Only {reachable_count} out of {N*N} cells are reachable.")
        return False

In [142]:
def find_smallest_fully_reachable_grid(moves, wrap_x=False, wrap_y=False):
    N = 2  # Start with 2 because 1x1 is trivial and 2x2 is the smallest non-trivial grid.
    while True:
        print(f"Checking grid size {N}x{N}...")
        if check_grid(N, moves, wrap_x, wrap_y):
            return N
        N += 1

In [143]:
base_moves = [(8,1), (7,4)]
all_moves = []
for dx,dy in base_moves:
    all_moves.extend([(dx,dy), (dx,-dy), (-dx,dy), (-dx,-dy), (dy,dx), (dy,-dx), (-dy,dx), (-dy,-dx)])

In [144]:
find_smallest_fully_reachable_grid(all_moves, wrap_x=False, wrap_y=False)

Checking grid size 2x2...
2x2: Only 1 out of 4 cells are reachable.
Checking grid size 3x3...
3x3: Only 1 out of 9 cells are reachable.
Checking grid size 4x4...
4x4: Only 1 out of 16 cells are reachable.
Checking grid size 5x5...
5x5: Only 1 out of 25 cells are reachable.
Checking grid size 6x6...
6x6: Only 1 out of 36 cells are reachable.
Checking grid size 7x7...
7x7: Only 1 out of 49 cells are reachable.
Checking grid size 8x8...
8x8: Only 1 out of 64 cells are reachable.
Checking grid size 9x9...
9x9: Only 1 out of 81 cells are reachable.
Checking grid size 10x10...
10x10: Only 1 out of 100 cells are reachable.
Checking grid size 11x11...
11x11: Only 1 out of 121 cells are reachable.
Checking grid size 12x12...
12x12: Only 1 out of 144 cells are reachable.
Checking grid size 13x13...
13x13: Only 1 out of 169 cells are reachable.
Checking grid size 14x14...
14x14: All cells are reachable.


14

In [145]:
find_smallest_fully_reachable_grid(all_moves, wrap_x=True, wrap_y=False)

Checking grid size 2x2...
2x2: Only 2 out of 4 cells are reachable.
Checking grid size 3x3...
3x3: All cells are reachable.


3

In [146]:
find_smallest_fully_reachable_grid(all_moves, wrap_x=False, wrap_y=True)

Checking grid size 2x2...
2x2: Only 2 out of 4 cells are reachable.
Checking grid size 3x3...
3x3: All cells are reachable.


3

In [147]:
find_smallest_fully_reachable_grid(all_moves, wrap_x=True, wrap_y=True)

Checking grid size 2x2...
2x2: All cells are reachable.


2

In [ ]:
#check_grid(14, all_moves, wrap_x=False, wrap_y=False, details=True)

# Conclusion

Fiddler Answer is  L = square root of 65 = 8.0622577482985496523666132303038

Extra Credit Answer is N = 14.

If we allow the board to be wrapped around a cylinder, so that one dimension wraps around, then even a 3x3 grid is sufficient for fully reachability.

And if we allow the board to be wrapped around a torus, so that both dimensions wrap around, then even a 2x2 grid is sufficient for fully reachability.

Even without wrapping, the 1x1 board is fully reachable, since there is just one starting cell, and no moves are needed.

But all these pedantry apart, the intended answer is clearly 14.